# Faster R-CNN: ignore-aware baseline on every fifth frame

We train COCO-pretrained Faster R-CNN ResNet-50 FPN V2 on every fifth training frame under the
same protocol as the RetinaNet experiments, so that the two detector families can be compared.
We use seed 2026, batch size 2, resize 640/1024, SGD with learning rate 0.0025, 100 warmup
updates, two step decays at two thirds and 90% of the updates, horizontal flips with
probability 0.5, mixed precision on CUDA and five epochs.

| Setting | RetinaNet runs | This run |
| --- | --- | --- |
| Training frames | 4,868, stride 5 within each clip | The same 4,868 frames |
| Model | RetinaNet ResNet-50 FPN V2 | Faster R-CNN ResNet-50 FPN V2 |
| Ignored regions | Sampled as background | Excluded from background sampling |
| Detections per image | 300 | 500 |
| Validation after every epoch | COCO maxDets 100 | COCO maxDets 100 and 500 |
| Reported checkpoint | Best epoch on validation | Final epoch, never chosen on validation |

We evaluate on all 2,846 official validation frames with the original category-specific COCO
crowd records. The competition's exact metric and ignore semantics remain unverified, so the
COCO values are a stand-in. We use no test data for model selection. Because this run changes
both the model and the ignored-region handling, its comparison with RetinaNet mixes the two
effects until a run with ignore handling switched off separates them.

**Stages.** Set `STAGE` in section 2 to `"probe"` for a short timing run that measures the real
cost, then to `"full"` for the experiment. Full training is the default. Set the stage to probe to measure throughput first.

## 1. Runtime

We record the actual GPU and software versions. The hosted CUDA environment uses its compatible
Python/PyTorch/TorchVision pair and needs Python 3.12 or newer; local development uses Python 3.14.

In [ ]:
import hashlib
import importlib
import importlib.util
import json
import logging
import math
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from zipfile import ZipFile

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12 or newer runtime.")

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()
requirements = {
    "polars": "polars>=1.31",
    "plotly": "plotly>=6",
    "pycocotools": "pycocotools>=2.0.11",
    "decouple": "python-decouple>=3.8",
    "PIL": "Pillow>=11",
    "tqdm": "tqdm>=4.67",
    "ipywidgets": "ipywidgets>=8",
    "googleapiclient": "google-api-python-client",
    "google.oauth2": "google-auth",
}
missing = [
    package for module, package in requirements.items() if importlib.util.find_spec(module) is None
]
if missing and IS_KAGGLE:
    uv = shutil.which("uv")
    if uv is None:
        installer = Path("/kaggle/working/install_uv.sh")
        urllib.request.urlretrieve("https://astral.sh/uv/install.sh", installer)
        subprocess.run(["sh", str(installer)], check=True)
        uv = str(Path.home() / ".local/bin/uv")
    subprocess.run(
        [uv, "pip", "install", "--system", "--python", sys.executable, *missing], check=True
    )

import polars as pl
import torch
import torchvision
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
LOGGER = logging.getLogger("faster_rcnn_kaggle")
if IS_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
pl.from_dicts(
    [
        {
            "Python": platform.python_version(),
            "PyTorch": torch.__version__,
            "TorchVision": torchvision.__version__,
            "CUDA": torch.version.cuda,
            "GPU": torch.cuda.get_device_name() if torch.cuda.is_available() else "None",
            "Free VRAM GiB": (
                torch.cuda.mem_get_info()[0] / 1024**3 if torch.cuda.is_available() else 0.0
            ),
        }
    ]
)

## 2. Data and output paths

On Kaggle, we load the project code from `hm2_retinanet/inputs/faster_rcnn_project.zip`
on Drive and read the attached competition data in place and keep prepared metadata and staged images
separate from experiment outputs. We use the same batch identifier across restarts so that a
compatible checkpoint resumes. Outputs go to `runs/faster_rcnn/<batch id>/<experiment>/`.

`config/faster_rcnn_baseline.toml` holds the recipe in tables. `[dataset]` has the frame stride,
the counts that the selection is checked against and `should_stage_images`. `[model]` overrides
`build_faster_rcnn` arguments, and omitted arguments keep the torchvision defaults. `[training]`
has the seed, epochs, batch size, learning rate, warmup updates, flip probability and checkpoint
interval. `[evaluation]` has the validation loader and the speed measurement. `[probe]` sizes the
timing run and `[export]` switches the test file. `config/faster_rcnn_tiny.toml` has the same
tables for a few frames on a CPU.

In [ ]:
if IS_KAGGLE:
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    UserSecretsClient = importlib.import_module("kaggle_secrets").UserSecretsClient
    token = json.loads(UserSecretsClient().get_secret("HM2_DRIVE_TOKEN"))
    if not token.get("refresh_token"):
        raise ValueError("HM2_DRIVE_TOKEN must contain refreshable Drive credentials.")
    credentials = Credentials.from_authorized_user_info(token)
    del token
    service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    drive_root = "16wH9gNp13kfa4T039nsiFcZwYOlEJI5Z"
    root_files = (
        service.files()
        .list(
            q=f"'{drive_root}' in parents and trashed = false",
            fields="files(id,name)",
            pageSize=1000,
        )
        .execute(num_retries=3)["files"]
    )
    inputs = [entry for entry in root_files if entry["name"] == "inputs"]
    if len(inputs) != 1:
        raise ValueError("Expected one inputs folder in hm2_retinanet on Drive.")
    input_files = (
        service.files()
        .list(
            q=f"'{inputs[0]['id']}' in parents and trashed = false",
            fields="files(id,name,size,md5Checksum)",
            pageSize=1000,
        )
        .execute(num_retries=3)["files"]
    )
    projects = [entry for entry in input_files if entry["name"] == "faster_rcnn_project.zip"]
    if len(projects) != 1:
        raise FileNotFoundError(
            "Upload exactly one faster_rcnn_project.zip to hm2_retinanet/inputs on Drive."
        )
    project = projects[0]
    archive_path = Path("/kaggle/temp/faster_rcnn_project.zip")
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive_path.with_suffix(".incoming")
    with temporary.open("wb") as stream:
        transfer = MediaIoBaseDownload(stream, service.files().get_media(fileId=project["id"]))
        has_completed = False
        while not has_completed:
            _, has_completed = transfer.next_chunk(num_retries=3)
    with temporary.open("rb") as stream:
        actual_md5 = hashlib.file_digest(stream, "md5").hexdigest()
    if actual_md5 != project["md5Checksum"] or temporary.stat().st_size != int(project["size"]):
        raise ValueError("Project archive transfer checksum mismatch.")
    temporary.replace(archive_path)
    REPOSITORY_ROOT = Path("/kaggle/temp/hm2_faster_rcnn")
    REPOSITORY_ROOT.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive_path) as archive:
        required_paths = {
            "src/hm2/faster_rcnn.py",
            "src/hm2/faster_rcnn_experiments.py",
            "src/hm2/drive_api.py",
            "config/faster_rcnn_baseline.toml",
        }
        if not required_paths.issubset(archive.namelist()):
            raise ValueError("The Drive archive is missing Faster R-CNN code or configuration.")
        for entry in archive.infolist():
            if (
                not (REPOSITORY_ROOT / entry.filename)
                .resolve()
                .is_relative_to(REPOSITORY_ROOT.resolve())
            ):
                raise ValueError("Project archive contains an unsafe path.")
        archive.extractall(REPOSITORY_ROOT)
    LOGGER.info("Downloaded and verified project code from Drive: %s", archive_path.name)
else:
    working_directory = Path.cwd()
    REPOSITORY_ROOT = (
        working_directory.parent
        if working_directory.name in {"notebooks", "experiments"}
        else working_directory
    )
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from decouple import config
from IPython.display import Markdown

from hm2.experiment_checkpoint import publish_snapshot, write_json
from hm2.faster_rcnn import compare_loss_windows
from hm2.faster_rcnn_experiments import (
    build_experiment_config,
    compute_source_identity,
    experiment_label,
    export_test_submission,
    find_competition_root,
    find_previous_output,
    load_experiment_settings,
    load_final_model,
    measure_inference_speed,
    prepare_experiment_tables,
    restore_previous_output,
    run_experiment,
    run_timing_probe,
    select_experiment_frames,
    select_probe_frames,
    stage_images,
    summarize_experiment,
    verify_selection,
)
from hm2.tiny_subset import flight_prefix, load_tiny_tables, prepare_tiny_subset

STAGE = "full"
STAGE = config("HM2_STAGE", default=STAGE)
if STAGE not in {"probe", "full"}:
    raise ValueError('STAGE must be "probe" or "full".')
RUN_MODE = "baseline" if IS_KAGGLE else "tiny"
EXPERIMENT = "baseline_stride5"
BATCH_ID = "faster_rcnn_stride5_5epochs_seed2026" if IS_KAGGLE else "faster_rcnn_tiny"
SESSION_HOURS = 11.0
OUTPUT_ROOT = Path("/kaggle/working") if IS_KAGGLE else REPOSITORY_ROOT
LOCAL_BATCH = OUTPUT_ROOT / "runs/faster_rcnn" / BATCH_ID
OUTPUT_DIRECTORY = LOCAL_BATCH / EXPERIMENT
PROBE_DIRECTORY = LOCAL_BATCH / "_timing_probe"
STAGING_ROOT = Path("/kaggle/temp/faster_rcnn_images")
SETTINGS = load_experiment_settings(REPOSITORY_ROOT / "config" / f"faster_rcnn_{RUN_MODE}.toml")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOCAL_BATCH.mkdir(parents=True, exist_ok=True)
{
    "Stage": STAGE,
    "Configuration": f"faster_rcnn_{RUN_MODE}.toml",
    "Batch": BATCH_ID,
    "Experiment": experiment_label(EXPERIMENT),
    "Local outputs": str(LOCAL_BATCH),
}

## 3. Dataset staging

We build the frame, target and ignored-region tables once from the original COCO files with the
shared table builder and keep them as parquet. We then copy only the images that the chosen stage
needs to fast local storage, because reading thousands of small files from the attached input is
slower. A stage that has copied a file before skips it.

In [ ]:
if IS_KAGGLE:
    SOURCE_ROOT = find_competition_root(KAGGLE_INPUT)
    frames, objects, ignored = prepare_experiment_tables(
        SOURCE_ROOT, Path("/kaggle/working/data/metadata")
    )
else:
    default_archive = Path.home() / "Downloads/multiple-object-video-detection-hw-2-ucu-ai-2026.zip"
    archive = Path(config("HM2_DATA_SOURCE", default=str(default_archive))).expanduser()
    tiny_selection = {
        split: SETTINGS["dataset"][f"{split}_frames"] for split in ("train", "val", "test")
    }
    SOURCE_ROOT = prepare_tiny_subset(
        archive, REPOSITORY_ROOT.parent / "hm2_tiny_data", tiny_selection
    )
    frames, objects, ignored = load_tiny_tables(SOURCE_ROOT)
selected, validation = select_experiment_frames(frames, SETTINGS["dataset"])
test_frames = frames.filter(pl.col("split") == "test")

IMAGE_ROOT = SOURCE_ROOT
staging = None
if SETTINGS["dataset"]["should_stage_images"]:
    if STAGE == "probe":
        probe_frames = select_probe_frames(selected, validation, ignored, SETTINGS)
        needed = [probe_frames["train"], probe_frames["ignore"], probe_frames["validation"]]
    else:
        needed = [selected, validation, test_frames]
    IMAGE_ROOT = STAGING_ROOT
    staging = stage_images(
        SOURCE_ROOT, IMAGE_ROOT, [path for part in needed for path in part["relative_path"]]
    )
pl.DataFrame(
    {
        "Item": ["Image root", "Staged files", "Copied megabytes", "Copy seconds"],
        "Value": [
            str(IMAGE_ROOT),
            str(staging["files"]) if staging else "0",
            f"{staging['copied_megabytes']:.0f}" if staging else "0",
            f"{staging['seconds']:.0f}" if staging else "0",
        ],
    }
)

## 4. Reproducibility and checkpoint checks

We verify the training selection against the expected counts and check that no flight appears in
both training and validation, because clips cut from one flight share a prefix such as
`uav0000013`. We identify the annotation files and the exact selection by hash. If a previous
notebook version is attached as input, we copy its checkpoint into the output folder so that the
run resumes; a checkpoint with a different recipe is rejected when training starts. The table
below records the recipe and the recovered progress.
On Kaggle, we verify Drive access and restore the latest compatible snapshot using the
`HM2_DRIVE_TOKEN` secret. The same secret also authorizes downloading the project archive.


In [ ]:
verify_selection(selected, validation, SETTINGS["dataset"])
selected_keys = selected["image_key"].to_list()
SOURCE_IDENTITY = compute_source_identity(SOURCE_ROOT, selected_keys)
EXPERIMENT_CONFIG = build_experiment_config(SETTINGS, EXPERIMENT, selected_keys, SOURCE_IDENTITY)

if STAGE == "full" and IS_KAGGLE:
    previous = find_previous_output(KAGGLE_INPUT, BATCH_ID, EXPERIMENT)
    if previous is not None and restore_previous_output(previous, OUTPUT_DIRECTORY):
        LOGGER.info("Restored the checkpoint of an attached previous version: %s", previous)
persist_checkpoint = None
if STAGE == "full" and IS_KAGGLE:
    from hm2.drive_api import (
        DriveApi,
        restore_checkpoint_from_drive_api,
        sync_checkpoint_to_drive_api,
        sync_results_to_drive_api,
        verify_drive_api_checkpoint,
    )

    api = DriveApi(service)
    batch_folder = api.folder(api.folder(drive_root, "batches"), BATCH_ID)
    run_folder = api.folder(batch_folder, EXPERIMENT)
    cache = Path("/kaggle/temp/drive_cache") / BATCH_ID
    quota = service.about().get(fields="storageQuota").execute(num_retries=3)["storageQuota"]
    if "limit" in quota and int(quota["limit"]) - int(quota.get("usage", 0)) < 2 * 1024**3:
        raise OSError("Keep at least 2 GiB free on Drive for checkpoint replacement.")
    probe_result = verify_drive_api_checkpoint(
        api, cache / "probe", api.folder(batch_folder, "_probe_api")
    )
    LOGGER.info("Drive checkpoint save and restore verified: %s", probe_result)
    progress = restore_checkpoint_from_drive_api(
        api, run_folder, OUTPUT_DIRECTORY, EXPERIMENT_CONFIG, cache / EXPERIMENT
    )
    LOGGER.info(
        "Drive progress: epoch=%s cursor=%s updates=%s",
        progress.get("epochs_completed", 0),
        progress.get("batch_cursor", 0),
        progress.get("global_step", 0),
    )

    def persist_checkpoint(path: Path) -> None:
        sync_checkpoint_to_drive_api(api, path, run_folder)


status_path = OUTPUT_DIRECTORY / "status.json"
restored = json.loads(status_path.read_text()) if status_path.is_file() else {}
model_settings = EXPERIMENT_CONFIG["model"]
pl.from_dicts(
    [
        {
            "Run": experiment_label(EXPERIMENT),
            "Training frames": selected.height,
            "Training flights": len({flight_prefix(clip) for clip in selected["clip"]}),
            "Validation frames": validation.height,
            "Epochs": EXPERIMENT_CONFIG["epochs"],
            "Batch size": EXPERIMENT_CONFIG["batch_size"],
            "Resize": f"{model_settings['min_size']}/{model_settings['max_size']}",
            "Ignore-aware": model_settings["is_ignore_aware"],
            "Detections per image": model_settings["detections_per_img"],
            "Epochs completed": restored.get("epochs_completed", 0),
            "Updates": restored.get("global_step", 0),
            "Completed": restored.get("is_completed", False),
        }
    ]
)

## 5. Timing probe

We train a few updates with the real recipe and time inference at batch size 1 and at the
validation batch size. We evaluate the probe predictions once and check that ignore-aware
sampling still excludes background and keeps positives under mixed precision. The probe saves
no checkpoint. Its projections are linear extrapolations from a few updates of an untrained
detector, not results.

In [ ]:
if STAGE == "probe":
    probe = run_timing_probe(
        IMAGE_ROOT, SOURCE_ROOT, frames, objects, ignored, PROBE_DIRECTORY, SETTINGS
    )
    batch_size = SETTINGS["evaluation"]["batch_size"]
    projection = probe["projection"]
    rows = {
        "Device": probe["gpu"] or probe["device"],
        "Training images per second": f"{probe['train_images_per_second']:.2f}",
        "Seconds per update": f"{probe['train_seconds_per_update']:.2f}",
        "Loader images per second": f"{probe['loader_images_per_second']:.1f}",
        "Peak GPU MiB": f"{probe['peak_gpu_mib']:.0f}" if probe["peak_gpu_mib"] else "None",
        "Inference ms per frame, batch 1": (
            f"{probe['inference']['batch_1']['milliseconds_per_frame']:.1f}"
        ),
        f"Inference ms per frame, batch {batch_size}": (
            f"{probe['inference'][f'batch_{batch_size}']['milliseconds_per_frame']:.1f}"
        ),
        "COCO evaluation seconds per frame": f"{probe['evaluation_seconds_per_frame']:.4f}",
        "Detections per frame (probe model)": f"{probe['detections_per_frame_mean']:.0f}",
        "Anchor negatives excluded": str(probe["ignore_check"]["anchor_negatives_excluded"]),
        "Proposal negatives excluded": str(probe["ignore_check"]["proposal_negatives_excluded"]),
        "Projected training minutes": f"{projection['train_minutes']:.0f}",
        "Projected validation minutes": f"{projection['validation_minutes']:.0f}",
        "Projected total minutes": f"{projection['total_minutes']:.0f}",
    }
    if staging and staging["megabytes_per_second"]:
        full_files = selected.height + validation.height + test_frames.height
        minutes = full_files * staging["megabytes"] / staging["files"]
        rows["Projected image copy minutes"] = (
            f"{minutes / staging['megabytes_per_second'] / 60:.1f}"
        )
    display(pl.DataFrame({"Measurement": list(rows), "Value": list(rows.values())}))

## 6. Training and validation

We run Faster R-CNN with ignore-aware sampling (every fifth frame). We save complete training state
locally at epoch boundaries and approximately every ten minutes. The model after the fifth epoch is
the result: we never choose an epoch on validation, and the per-epoch validation is recorded only as
information. A session time limit pauses execution without changing the five-epoch budget. GPU kernels
and changes of runtime can affect numerical reproducibility.

After the last epoch we time inference on 300 validation frames at batch size 1 and write a test
submission file. The file stays unsubmitted, and the official sample file is not part of the data,
so only the row membership of the test list is verified.
Completed snapshots are synchronized to Drive after checksum verification. The previous verified
generation is retained until its replacement is confirmed.


In [ ]:
status: dict = {"is_completed": False}
model = None
if STAGE == "full":
    status = run_experiment(
        IMAGE_ROOT,
        SOURCE_ROOT,
        frames,
        objects,
        ignored,
        OUTPUT_DIRECTORY,
        SETTINGS,
        EXPERIMENT,
        seconds=SESSION_HOURS * 3600,
        on_checkpoint=persist_checkpoint,
    )
    display(
        pl.from_dicts(
            [
                {
                    "Run": experiment_label(EXPERIMENT),
                    "Epochs completed": status["epochs_completed"],
                    "Updates": status["global_step"],
                    "Image exposures": status["image_exposures"],
                    "Completed": status["is_completed"],
                }
            ]
        )
    )
    if status["is_stopped"]:
        raise InterruptedError(f"Remove {OUTPUT_DIRECTORY / 'STOP'} before resuming.")
    if not status["is_completed"]:
        LOGGER.info(
            "Session budget reached. Rerun with the same batch and configuration to resume."
        )

In [ ]:
if STAGE == "full" and status["is_completed"]:
    model, _ = load_final_model(OUTPUT_DIRECTORY, DEVICE)
    speed = measure_inference_speed(
        model, IMAGE_ROOT, validation, objects, ignored, SETTINGS["evaluation"], DEVICE
    )
    write_json(OUTPUT_DIRECTORY / "speed.json", speed)
    if SETTINGS["export"]["should_export_test_predictions"]:
        submission = export_test_submission(
            model, IMAGE_ROOT, frames, OUTPUT_DIRECTORY, SETTINGS["evaluation"], DEVICE
        )
        display(
            pl.DataFrame(
                [
                    {
                        "File": submission["file"],
                        "Images": submission["images"],
                        "Detections": submission["detections"],
                        "Empty images": submission["empty_images"],
                        "Official sample verified": submission["official_sample_verified"],
                    }
                ]
            )
        )
if STAGE == "full" and persist_checkpoint is not None and status["is_completed"]:
    publish_snapshot(OUTPUT_DIRECTORY)
    persist_checkpoint(OUTPUT_DIRECTORY)
    sync_results_to_drive_api(api, OUTPUT_DIRECTORY, run_folder)

## 7. Saved progress

We inspect the saved records of the run. After a runtime reset, we rerun the stage `"full"` with the same batch identifier and recipe. On Kaggle,
we restore the last verified checkpoint from Google Drive. Work after
the last checkpoint may repeat; completed epochs and their validation are retained.

We review the comparison, per-class AP, loss curves and timing in
[the results notebook](../notebooks/05_faster_rcnn_results.ipynb) after downloading the output
folder `runs/faster_rcnn/<batch id>/` into the repository. We treat metric differences as
descriptive single-seed results.

In [ ]:
summary = summarize_experiment(OUTPUT_DIRECTORY)
display(
    pl.DataFrame(
        [
            {
                "Run": experiment_label(EXPERIMENT),
                "Status": summary["status"],
                "Epochs completed": summary.get("epochs_completed", 0),
                "Updates": summary.get("global_step", 0),
                "Training minutes": round(summary.get("wall_train_seconds", 0.0) / 60, 1),
            }
        ]
    )
)
history_path = OUTPUT_DIRECTORY / "history.json"
if summary["status"] == "Completed" and history_path.is_file():
    steps_per_epoch = math.ceil(selected.height / SETTINGS["training"]["batch_size"])
    loss_check = compare_loss_windows(json.loads(history_path.read_text()), steps_per_epoch)
    display(
        pl.DataFrame(
            {
                "Mean loss": ["First epoch", "Last epoch", "Last divided by first"],
                "Value": [
                    round(loss_check["first_window_loss"], 4),
                    round(loss_check["last_window_loss"], 4),
                    round(loss_check["ratio"], 3),
                ],
            }
        )
    )